In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder \
.appName("User Defined Functions") \
.master("local[2]") \
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/09/28 18:20:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read \
.option("header",True) \
.option("inferSchema", True) \
.option("compression","gzip") \
.csv("file:///opt/examples/datasets/Hotel_Reviews.csv.gz")

In [4]:
df2 = df.withColumn("Tags", 
                     F.split(F.col("Tags"), ",")
                     .cast(ArrayType(StringType()))) \
.withColumn("Review_Date", F.to_date(F.col("Review_Date"),"M/d/yyyy"))

In [5]:
df2.limit(2).toPandas()

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,2017-08-03,7.7,Hotel Arena,Russia,I am so angry that i made this post available...,397,1403,Only the park outside of the hotel was beauti...,11,7,2.9,"[[' Leisure trip ', ' Couple ', ' Duplex Dou...",0 days,52.3605759,4.9159683
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,2017-08-03,7.7,Hotel Arena,Ireland,No Negative,0,1403,No real complaints the hotel was great great ...,105,7,7.5,"[[' Leisure trip ', ' Couple ', ' Duplex Dou...",0 days,52.3605759,4.9159683


# Define Function

In [6]:
def upper_case(x):
    return x.upper()

In [7]:
upper_case("ali gel")

'ALI GEL'

# Register UDF

In [8]:
from pyspark.sql.types import StringType

In [9]:
upper_case_udf = F.udf(upper_case, StringType())

In [10]:
df.withColumn("Nationality_Upper", upper_case_udf("Reviewer_Nationality")).select("Reviewer_Nationality","Nationality_Upper").show(4)

+--------------------+-----------------+
|Reviewer_Nationality|Nationality_Upper|
+--------------------+-----------------+
|             Russia |          RUSSIA |
|            Ireland |         IRELAND |
|          Australia |       AUSTRALIA |
|     United Kingdom |  UNITED KINGDOM |
+--------------------+-----------------+
only showing top 4 rows



# UDF on multiple columns

In [11]:
def hotel_and_country(hotel, country):
    return "{} - {}".format(hotel, country)

In [12]:
hotel_and_country_udf = spark.udf.register("hotel_and_country_udf", 
                                           hotel_and_country, StringType())

In [13]:
df.select(hotel_and_country_udf("Hotel_Name","Reviewer_Nationality") \
          .alias("Hotel_Country")).show(truncate=False)

+------------------------------+
|Hotel_Country                 |
+------------------------------+
|Hotel Arena -  Russia         |
|Hotel Arena -  Ireland        |
|Hotel Arena -  Australia      |
|Hotel Arena -  United Kingdom |
|Hotel Arena -  New Zealand    |
|Hotel Arena -  Poland         |
|Hotel Arena -  United Kingdom |
|Hotel Arena -  United Kingdom |
|Hotel Arena -  Belgium        |
|Hotel Arena -  Norway         |
|Hotel Arena -  United Kingdom |
|Hotel Arena -  France         |
|Hotel Arena -  United Kingdom |
|Hotel Arena -  Italy          |
|Hotel Arena -  Canada         |
|Hotel Arena -  Italy          |
|Hotel Arena -  United Kingdom |
|Hotel Arena -  Ireland        |
|Hotel Arena -  Netherlands    |
|Hotel Arena -  Australia      |
+------------------------------+
only showing top 20 rows



In [14]:
df1 = df.withColumn("hotel_country", hotel_and_country_udf("Hotel_Name","Reviewer_Nationality")) \
.drop("Hotel_Name", "Reviewer_Nationality")

In [15]:
df1.printSchema()

root
 |-- Hotel_Address: string (nullable = true)
 |-- Additional_Number_of_Scoring: integer (nullable = true)
 |-- Review_Date: string (nullable = true)
 |-- Average_Score: double (nullable = true)
 |-- Negative_Review: string (nullable = true)
 |-- Review_Total_Negative_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews: integer (nullable = true)
 |-- Positive_Review: string (nullable = true)
 |-- Review_Total_Positive_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews_Reviewer_Has_Given: integer (nullable = true)
 |-- Reviewer_Score: double (nullable = true)
 |-- Tags: string (nullable = true)
 |-- days_since_review: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lng: string (nullable = true)
 |-- hotel_country: string (nullable = true)



# Pandas UDFs

    One of the previous prevailing issues with using PySpark UDFs was that they had
    slower performance than Scala UDFs. This was because the PySpark UDFs required
    data movement between the JVM and Python, which was quite expensive. To resolve
    this problem, Pandas UDFs (also known as vectorized UDFs) were introduced as part
    of Apache Spark 2.3. A Pandas UDF uses Apache Arrow to transfer data and Pandas
    to work with the data. You define a Pandas UDF using the keyword pandas_udf as
    the decorator, or to wrap the function itself. Once the data is in Apache Arrow format,
    there is no longer the need to serialize/pickle the data as it is already in a format
    consumable by the Python process. Instead of operating on individual inputs row by
    row, you are operating on a Pandas Series or DataFrame (i.e., vectorized execution).

In [16]:
import pandas as pd

In [17]:
def upper_case(col: pd.Series) -> pd.Series:
    return col.transform(lambda x: x.upper())

In [18]:
# Format
# F.pandas_udf(function, returnType)

In [21]:
upper_case_pdudf = F.pandas_udf(upper_case, returnType=StringType())

In [20]:
! pip install pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 2.3 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 23.2.1
[notice] To update, run: pip install --upgrade pip


In [22]:
df.select("Hotel_Name",upper_case_pdudf("Hotel_Name").alias("Hotel_Name_Upper")).show(5)

+-----------+----------------+
| Hotel_Name|Hotel_Name_Upper|
+-----------+----------------+
|Hotel Arena|     HOTEL ARENA|
|Hotel Arena|     HOTEL ARENA|
|Hotel Arena|     HOTEL ARENA|
|Hotel Arena|     HOTEL ARENA|
|Hotel Arena|     HOTEL ARENA|
+-----------+----------------+
only showing top 5 rows



# pandas udf on multiple columns

In [23]:
def pd_hotel_and_country(hotel:pd.Series, country:pd.Series) -> pd.Series:
    return hotel + ' - ' + country

In [24]:
pd_hotel_and_country_udf = F.pandas_udf(pd_hotel_and_country, StringType())

In [25]:
df2.select(pd_hotel_and_country_udf("Hotel_Name", "Reviewer_Nationality") \
           .alias("HC")).show(n=4, truncate=False)

+------------------------------+
|HC                            |
+------------------------------+
|Hotel Arena -  Russia         |
|Hotel Arena -  Ireland        |
|Hotel Arena -  Australia      |
|Hotel Arena -  United Kingdom |
+------------------------------+
only showing top 4 rows



In [26]:
spark.stop()